In [1]:
!pip install --quiet docling


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os 
import requests

In [3]:
import os

local_file = r"C:\Ask My Paper\Data\Copy of AI Engineering-25-125.pdf"

if os.path.exists(local_file):
    print(f"PDF found: {local_file}")
else:
    print("PDF file not found.")

PDF found: C:\Ask My Paper\Data\Copy of AI Engineering-25-125.pdf


Now that the file is loaded in our local folder, we can use Docling to parse
and analyze it.

In [7]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat


pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True
res = DocumentConverter( format_options={ InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options), }).convert(local_file)
doc = res.document

[INFO] 2026-08-19 14:07:05,498 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-19 14:07:05,506 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-19 14:07:05,506 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.2/torch/PP-OCRv6/det/PP-OCRv6_det_small.pth
[INFO] 2026-08-19 14:07:09,278 [RapidOCR] download_file.py:82: Download size: 9.77MB
[INFO] 2026-08-19 14:07:11,261 [RapidOCR] download_file.py:95: Successfully saved to: C:\Ask My Paper\myenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-19 14:07:11,272 [RapidOCR] main.py:50: Using C:\Ask My Paper\myenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-19 14:07:11,414 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-19 14:07:11,416 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-19 14:07:11,416 [RapidOCR] download_file.py:68: Initiating download: https://www.models

In [12]:
table = doc.tables[1]
table_df = table.export_to_dataframe()
table_df

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


,Group,Occupations with highest exposure,% Exposure
0,Human α,Interpreters and translators Survey researcher...,76.5 75.0 68.8 66.7 66.7
1,Human β,Survey researchers Writers and authors Interpr...,84.4 82.5 82.4 80.6 77.8
2,Human ζ,Mathematicians Tax preparers Financial quantit...,100.0 100.0 100.0 100.0 100.0


In [13]:
print(table_df.shape)

(3, 3)


In [15]:
def format_context_item(item, max_rows=100): 
    """
    Formats context items dynamically, handling strings, DataFrames, 
    and nested JSON with safety checks for production.
    """ 
    # 1. Handle standard text chunks 
    if isinstance(item, str): 
        return f"- FACT: {item}" 
     
    # 2. Handle DataFrames (dynamic schema) 
    if isinstance(item, pd.DataFrame): 
        # If the table is too long, we truncate it but keep the headers 
        if len(item) > max_rows: 
            item = item.head(max_rows) 
            truncation_note=f"\n[Note: Table truncated to first {max_rows} rows]" 
        else: 
            truncation_note = "" 
             
        return f"""- DATAFRAME (Columns: {list(item.columns)}):
                {item.to_json(orient='records', indent=2)}{truncation_note}""" 
 
    # 3. Handle generic structured data (JSON/Dict) 
    try: 
        serialized = json.dumps(item, indent=2) 
        return f"- STRUCTURED DATA:\n{serialized}" 
    except (TypeError, ValueError): 
        # Fallback if the data isn't JSON-serializable 
        return f"- DATA (Raw): {str(item)}" 
 
 
formatted_context = "\n".join(format_context_item(c) for c in context)
prompt = f"""
You are an assistant for question-answering tasks. Use the following pieces of 
retrieved context to answer the question. If you don't know the answer, just 
say 
that you don't know. 
<question>
{question}
</question> 
<context>
{formatted_context}
</context> 
Answer:
"""

NameError: name 'context' is not defined